# Error Analysis

This notebook contains the error analysis for the random forest model. We will evaluate the model with and without the email domains included.

Step One: Create a new dataset with target value, predicted values for each model, and the difference between the target value and the predicted values as well.

In [81]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score
from scipy.stats import binomtest

In [24]:
# import data
X_train = pd.read_csv("train_transaction.csv")
X_val = pd.read_csv("val_transaction.csv")

# Split target column from main data
y_train = X_train['isFraud']
X_train.drop(['isFraud'], axis=1, inplace=True)

y_val = X_val['isFraud']
X_val.drop(['isFraud'], axis=1, inplace=True)

# drop unnecessary columns
X_train.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
X_val.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)

# create seperate train/val set without email columns
X_train_v2 = X_train[X_train.columns.drop(list(X_train.filter(regex='email')))]
X_val_v2 = X_val[X_val.columns.drop(list(X_val.filter(regex='email')))]

In [25]:
# build model with best hyperparameters
classifier = RandomForestClassifier(
    n_estimators=195,
    max_depth=48,
    min_samples_split=10,
    min_samples_leaf=2,
    min_weight_fraction_leaf=0.0,
    max_samples=1.0
)

## Email Model

In [50]:
# build email model
model_with = classifier.fit(X_train, y_train)
y_pred = model_with.predict(X_val)

proba_email = model_with.predict_proba(X_val)[:,1]
label_email = model_with.predict(X_val)

## Non-Email Model

In [49]:
# build non-email model
model_wo = classifier.fit(X_train_v2, y_train)
y_pred_v2 = model_wo.predict(X_val_v2)

proba_wo_email= model_wo.predict_proba(X_val_v2)[:,1]
label_wo_email = model_wo.predict(X_val_v2)

## Building the New Dataset

In [70]:
error_analysis_df = X_val.copy()
error_analysis_df['email_proba'] = proba_email
error_analysis_df['email_label'] = label_email
error_analysis_df['proba_wo_email'] = proba_wo_email
error_analysis_df['label_wo_email'] = label_wo_email
error_analysis_df['difference'] = error_analysis_df['email_proba'] - error_analysis_df['proba_wo_email']
error_analysis_df['model_match'] = error_analysis_df['label_wo_email'] == error_analysis_df['email_label']
error_analysis_df['true_y'] = y_val

error_analysis_df.head()

,TransactionID,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,...,M4_M1,M4_M2,M4_nan,email_proba,email_label,proba_wo_email,label_wo_email,difference,model_match,true_y
0,3030130,1035237,34.50,16163,551.0,150.0,226.0,325.0,87.0,2.0,...,0.0,0.0,0.0,0.040298,0,0.062644,0,-0.022346,True,0
1,3059303,1618570,226.00,15813,251.0,150.0,226.0,441.0,87.0,NaN,...,0.0,0.0,0.0,0.031891,0,0.041740,0,-0.009850,True,0
2,3280354,7243463,108.50,15497,490.0,150.0,226.0,299.0,87.0,30.0,...,0.0,0.0,0.0,0.002769,0,0.001066,0,0.001703,True,0
3,3019957,831523,159.95,12544,321.0,150.0,226.0,184.0,87.0,NaN,...,0.0,0.0,1.0,0.000090,0,0.001026,0,-0.000936,True,0
4,3001777,414969,77.95,17188,321.0,150.0,226.0,299.0,87.0,NaN,...,0.0,0.0,1.0,0.006904,0,0.002002,0,0.004902,True,0


In [71]:
## ----- find the top 20 disagreements between models -----
top_disagreements = error_analysis_df.reindex(
    error_analysis_df["difference"].abs().sort_values(ascending=False).index
)
print("Top 20 rows where emails changed the prediction most:")
top_disagreements.head(20)


Top 20 rows where emails changed the prediction most:


,TransactionID,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,...,M4_M1,M4_M2,M4_nan,email_proba,email_label,proba_wo_email,label_wo_email,difference,model_match,true_y
1798,3156948,3661750,75.000,16661,490.0,150.0,226.0,327.0,87.0,NaN,...,0.0,0.0,1.0,0.443951,0,0.098701,0,0.345250,True,1
25576,3178710,4304382,149.015,9633,296.0,185.0,138.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.386488,0,0.641239,1,-0.254751,False,1
27097,3131121,2962072,50.000,12839,321.0,150.0,226.0,469.0,87.0,NaN,...,0.0,0.0,1.0,0.232696,0,0.026922,0,0.205773,True,1
19545,3037736,1198114,37.000,5853,225.0,150.0,117.0,181.0,87.0,NaN,...,0.0,0.0,0.0,0.456790,0,0.256278,0,0.200513,True,1
8573,3163441,3820189,50.000,4090,490.0,150.0,226.0,310.0,87.0,NaN,...,0.0,0.0,1.0,0.266892,0,0.073605,0,0.193287,True,0
26373,3138133,3108965,50.000,18227,583.0,150.0,226.0,330.0,87.0,NaN,...,0.0,0.0,1.0,0.300232,0,0.124436,0,0.175795,True,1
40207,3058157,1606845,15.566,14276,177.0,185.0,137.0,NaN,NaN,NaN,...,0.0,1.0,0.0,0.494255,0,0.667908,1,-0.173653,False,1
13132,3191422,4671891,83.602,17433,274.0,185.0,137.0,NaN,NaN,NaN,...,0.0,1.0,0.0,0.441992,0,0.271337,0,0.170654,True,0
48061,3147593,3374026,50.000,1724,583.0,150.0,226.0,126.0,87.0,NaN,...,0.0,0.0,1.0,0.241574,0,0.074884,0,0.166690,True,0
38956,3031549,1075317,200.000,2616,327.0,150.0,102.0,126.0,87.0,NaN,...,0.0,0.0,1.0,0.068161,0,0.233733,0,-0.165572,True,0


For the top disagreements, both models typically misclassify the true y value. However, the label using the email is more likely to classify a value as not fraud when it really is fraud (false negative), which is extremely dangerous in fraud reporting. In the two cases where the models had different classifications (model_match), the model without the email classified the true value correctly. Both models typically misclassified the same values, however.

In [77]:
def categorize(row):
  if row["label_wo_email"] == row["email_label"]:
    return "agree"
  if row["email_label"] == row["true_y"]:
    return "email_helped"
  else:
    return "email_hurt"

top_disagreements["disagreement_type"] = top_disagreements.apply(categorize, axis=1)
print("\nDisagreement Table:")
top_disagreements["disagreement_type"].value_counts()



Disagreement Table:


,count
disagreement_type,
agree,49025
email_hurt,29
email_helped,17


While adding the email colums back in didn't have much of an effect, it hurt more than it helped.

In [83]:
correct_wo = (error_analysis_df["label_wo_email"] == error_analysis_df["true_y"])
correct_with = (error_analysis_df["email_label"] == error_analysis_df["true_y"])

# b = columns where having the emails hurt without
b = int(((correct_wo) & (~correct_with)).sum())
# c = having the emails helped
c = int(((~correct_wo) & (correct_with)).sum())

print(f"\nMcNemar's test: b={b} (without-correct/with-wrong), "
      f"c={c} (without-wrong/with-correct)")

# perform two-sided test of significance
p = binomtest(min(b, c), b + c, 0.5).pvalue * 2
p = min(p, 1.0)
print(f"Approx. two-sided p-value: {p:.4f}")




McNemar's test: b=29 (without-correct/with-wrong), c=17 (without-wrong/with-correct)
Approx. two-sided p-value: 0.2076


Adding or removing the emails to the dataset does not significantly help or hurt the analysis (p=0.2 > p=0.05).

Seeing as there is no significant effect on the data, we do not feel the need to attribute predictions to these features or measure their effect in another way.